# AFT 实验结果可视化与表格生成
此脚本会自动读取 `aft/` 文件夹下的所有 JSON 结果文件，计算均值，并生成对比表格。

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# 寻找所有的 aft 结果文件
files = glob.glob('aft/*.json')
if not files:
    print("在 aft/ 文件夹下没有找到 JSON 文件。请先运行 exp2_run_parallel_aft.ipynb")

for filename in files:
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    res = data['results']
    params = data['parameters']
    N = len(res)
    
    print(f"================ 文件: {os.path.basename(filename)} ================")
    md_table = f"### {params.get('noise_type', 'normal')} 噪声下的性能对比 (AFT, N={N})\n"
    md_table += "| Method | RMSE | MAE | Kendall_Correlation | Pearson_Correlation | Time (s) |\n"
    md_table += "|---|---|---|---|---|---|\n"
    
    methods = ['Pooled', 'Local', 'Avg', 'D-subGD', 'Proposed']
    
    avg_hist = None
    pooled_hist = None
    dgd_hist = None
    avg_avg_rmse = None
    avg_local_rmse = None
    avg_pooled_rmse = None
    avg_dgd_rmse = None
    
    for method in methods:
        # 检查该方法是否在结果中存在 (用户可能设置为 False 跳过了)
        if method in res[0]:
            rmses = [r[method]['RMSE'] for r in res]
            maes = [r[method]['MAE'] for r in res]
            kendalls = [r[method].get('Kendall_Correlation', 0.0) for r in res]
            pearsons = [r[method].get('Pearson_Correlation', 0.0) for r in res]
            times = [r[method]['Time'] for r in res]
            
            mean_rmse = np.mean(rmses)
            mean_mae = np.mean(maes)
            mean_kendall = np.mean(kendalls)
            mean_pearson = np.mean(pearsons)
            mean_time = np.mean(times)
            
            md_table += f"| {method} | {mean_rmse:.4f} | {mean_mae:.4f} | {mean_kendall:.4f} | {mean_pearson:.4f} | {mean_time:.2f} |\n"
            
            if method == 'Proposed' and 'hist_rmse' in res[0]['Proposed']:
                avg_hist = np.mean([r['Proposed']['hist_rmse'] for r in res], axis=0)
            if method == 'Pooled' and 'hist_rmse' in res[0]['Pooled']:
                pooled_hist = np.mean([r['Pooled']['hist_rmse'] for r in res], axis=0)
            if method == 'D-subGD' and 'hist_rmse' in res[0]['D-subGD']:
                dgd_hist = np.mean([r['D-subGD']['hist_rmse'] for r in res], axis=0)
            if method == 'Avg':
                avg_avg_rmse = mean_rmse
            if method == 'Local':
                avg_local_rmse = mean_rmse
            if method == 'Pooled':
                avg_pooled_rmse = mean_rmse
            if method == 'D-subGD':
                avg_dgd_rmse = mean_rmse
                
    print(md_table)
    print("====================================================================\n")
    
    if avg_hist is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(avg_hist, marker='o', label='Proposed RMSE (Avg)')
        total_steps = len(avg_hist) - 1
        if avg_avg_rmse is not None:
            plt.hlines(avg_avg_rmse, xmin=0, xmax=total_steps, color='r', linestyle='--', label=f'Avg MR (RMSE={avg_avg_rmse:.4f})')
        if avg_local_rmse is not None:
            plt.hlines(avg_local_rmse, xmin=0, xmax=total_steps, color='g', linestyle='-.', label=f'Local MR (RMSE={avg_local_rmse:.4f})')
        if pooled_hist is not None:
            x_pooled = np.arange(len(pooled_hist))
            plt.plot(x_pooled, pooled_hist, color='m', linestyle=':', label=f'Pooled MR')
        elif avg_pooled_rmse is not None:
            plt.hlines(avg_pooled_rmse, xmin=0, xmax=total_steps, color='m', linestyle=':', label=f'Pooled MR (RMSE={avg_pooled_rmse:.4f})')
        if dgd_hist is not None:
            x_dgd = np.arange(len(dgd_hist))
            plt.plot(x_dgd, dgd_hist, color='c', linestyle='-', label=f'D-subGD')
        elif avg_dgd_rmse is not None:
            plt.hlines(avg_dgd_rmse, xmin=0, xmax=total_steps, color='c', linestyle='-', label=f'D-subGD (RMSE={avg_dgd_rmse:.4f})')
        plt.xlabel('Outer Iteration t')
        plt.ylabel('RMSE')
        plt.title(f"Convergence of Proposed (AFT) - {params.get('noise_type', 'normal')} noise")
        plt.legend()
        plt.grid(True)
        plt.show()
